# Stage-B Ensemble v2 — α-Diverse Pool with Learned Combination

**Extends Stage 6 (Model Improvement).** Follows `two_stage_stageB_des.ipynb`, which was a negative
result, and is designed against its two diagnosed causes.

## What failed in v1, and what changes here

| v1 diagnosis | v1 evidence | v2 response |
|---|---|---|
| **No diversity in the pool** | XGB/RF/ET disagree on only 13.2% of rows; XGB right 44.8% vs RF 29.6% when they do — dominance, not complementarity | Build diversity **along the α axis**, where the weighting study already proved inverted profiles exist |
| **Competence estimator too noisy** | per-instance weights sat at 0.333 ± 0.02 — a static ensemble in disguise; picked ET (weakest model) on 44% of rows | **Learn** the combination (per-class weights / meta-learner) instead of estimating it from a local kNN |

The key observation is that v1 searched for complementarity between *algorithms* and found none, but
the weighting study already contains it between *α values*:

| Base | macro-P | macro-R |
|---|---|---|
| uniform (α=0) | 0.685 | 0.570 |
| α=0.25 | 0.659 | 0.597 |
| α=0.75 | 0.578 | 0.637 |
| `balanced` (α=1) | 0.545 | 0.655 |

A 14-point precision↔recall swing across the pool — genuinely different decision boundaries,
especially on the rare families where the weighting bites hardest. That is the diversity an ensemble
needs, and v1 did not have it.

## Why a *learned* combiner

Per-instance local accuracy has to estimate competence from ~30 neighbours, which is why its weights
collapsed to uniform. A **per-class weight matrix** $W \in \mathbb{R}^{M \times C}$ has only
$M \times C \approx 63$ parameters fitted against thousands of rows, and it varies the weighting exactly
where the models actually differ — by class. It is still "dynamic" in the sense that matters (the
weighting is not one global constant), but with far lower variance.

> **Protocol.** Base models train on `train`. The combiner is fitted on **DSEL**, which the base
> models never saw. Variant selection happens on **SEL**. The test set is touched once. Base models
> stay train-only, so the like-for-like reference is the train-only single XGBoost (0.6125), not the
> 0.6157 that was refit on train+val.

## 1. Setup, splits, gate

Identical seeds to the weighting and DES notebooks, so every split matches row-for-row.

In [1]:
%pip install xgboost imbalanced-learn -q

import numpy as np, pandas as pd, time, warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
RS = 42

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun   = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)
X_tr,  X_val,  y_tr,  y_val  = train_test_split(X_trf, y_trf, test_size=0.2, random_state=RS, stratify=y_trf)
X_dsel, X_sel, y_dsel, y_sel = train_test_split(X_val, y_val, test_size=0.5, random_state=RS, stratify=y_val)

def make_xgb(**kw):
    p = dict(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9,
             colsample_bytree=0.9, tree_method="hist", eval_metric="mlogloss",
             n_jobs=-1, random_state=RS)
    p.update(kw)
    return XGBClassifier(**p)

gate = make_xgb()
gate.fit(X_tr, (y_tr != 0).astype(int),
         sample_weight=compute_sample_weight("balanced", (y_tr != 0).astype(int)))
gate_sel, gate_test = gate.predict(X_sel), gate.predict(X_test)

attack_labels = sorted(l for l in y_tr.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
idx_to_fam = {i: f for f, i in fam_to_idx.items()}
NC  = len(attack_labels)
FAM = [LABELS[idx_to_fam[i]] for i in range(NC)]

def attack_only(Xa, ya):
    m = ya != 0
    return Xa[m].reset_index(drop=True), ya[m].map(fam_to_idx).reset_index(drop=True)

X_tr_att,   y_tr_att   = attack_only(X_tr,   y_tr)
X_dsel_att, y_dsel_att = attack_only(X_dsel, y_dsel)
X_sel_att,  y_sel_att  = attack_only(X_sel,  y_sel)
print("train attack %d | DSEL attack %d | SEL attack %d | test %d"
      % (len(y_tr_att), len(y_dsel_att), len(y_sel_att), len(y_test)))

Note: you may need to restart the kernel to use updated packages.
train attack 51949 | DSEL attack 6494 | SEL attack 6493 | test 61235


## 2. The α-diverse base pool

Seven members. Five are the *same* XGBoost at different points on the α curve — deliberately spanning
the precision↔recall axis rather than trying to be individually optimal. The capped-SMOTE member adds
a resampling-induced bias (a different mechanism, not just a different weight), and RF/ET add
algorithmic variety on top.

In [2]:
def sample_w(y_arr, wc):
    w = wc[y_arr].astype(float)
    return w * len(w) / w.sum()

def wc_alpha(n, a):
    w = (n.sum() / n) ** a
    return w / w.mean()

n_tr = np.bincount(y_tr_att.values, minlength=NC)
WC25 = wc_alpha(n_tr, 0.25)

def fit_pool():
    pool = {}
    for a in (0.0, 0.25, 0.5, 1.0):
        t = time.time()
        m = make_xgb(); m.fit(X_tr_att, y_tr_att, sample_weight=sample_w(y_tr_att.values, wc_alpha(n_tr, a)))
        pool[f"XGB a={a}"] = m
        print("  XGB a=%-5s %4.0fs" % (a, time.time() - t))
    strat = {c: 2000 for c in range(NC) if n_tr[c] < 2000}
    Xs, ys = SMOTE(random_state=RS, sampling_strategy=strat).fit_resample(X_tr_att, y_tr_att)
    m = make_xgb(); m.fit(Xs, ys); pool["XGB smote2000"] = m
    print("  XGB smote2000 done")
    for nm, mdl in (("RF a=0.25", RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RS)),
                    ("ET a=0.25", ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RS))):
        mdl.fit(X_tr_att, y_tr_att, sample_weight=sample_w(y_tr_att.values, WC25))
        pool[nm] = mdl
        print("  %s done" % nm)
    return pool

t = time.time(); POOL = fit_pool(); MODELS = list(POOL); M = len(MODELS)
print("pool of %d fitted in %.0fs" % (M, time.time() - t))

def probas_of(Xq):
    return [POOL[k].predict_proba(Xq) for k in MODELS]

P_dsel = probas_of(X_dsel_att)     # combiner is FITTED here
P_sel  = probas_of(X_sel)          # all rows -- routing needs a family for every gate-positive row
P_test = probas_of(X_test)
print("probabilities computed")

  XGB a=0.0     10s
  XGB a=0.25     9s
  XGB a=0.5      9s
  XGB a=1.0      9s
  XGB smote2000 done
  RF a=0.25 done
  ET a=0.25 done
pool of 7 fitted in 52s
probabilities computed


## 3. Did the α axis actually buy diversity?

The check v1 failed. Three things to look for: **inverted profiles** across the pool, a **higher
disagreement rate** than v1's 13.2%, and a **higher oracle bound** than v1's 0.6799.

In [3]:
yd = y_dsel_att.values
pd_pred = [p.argmax(1) for p in P_dsel]

print("--- individual members on DSEL (Stage-B) ---")
print(pd.DataFrame([dict(model=nm,
        mF1=round(f1_score(yd, pr, average="macro", zero_division=0), 4),
        mP =round(precision_score(yd, pr, average="macro", zero_division=0), 4),
        mR =round(recall_score(yd, pr, average="macro", zero_division=0), 4))
    for nm, pr in zip(MODELS, pd_pred)]).to_string(index=False))

D = np.zeros((M, M))
for i in range(M):
    for j in range(M):
        D[i, j] = (pd_pred[i] != pd_pred[j]).mean()
print("\n--- pairwise disagreement rate (v1 pool averaged ~0.12) ---")
print(pd.DataFrame(D, index=MODELS, columns=MODELS).round(3).to_string())
print("mean off-diagonal disagreement: %.4f" % D[~np.eye(M, dtype=bool)].mean())

any_right = np.zeros(len(yd), bool)
for pr in pd_pred:
    any_right |= (pr == yd)
oracle = np.where(any_right, yd, pd_pred[0])
best_single = max(f1_score(yd, pr, average="macro", zero_division=0) for pr in pd_pred)
orc = f1_score(yd, oracle, average="macro", zero_division=0)
print("\noracle macro-F1 %.4f | best single %.4f | headroom %.4f | any-correct %.2f%%"
      % (orc, best_single, orc - best_single, 100*any_right.mean()))
print("(v1 pool: oracle 0.6799, headroom 0.0804)")

--- individual members on DSEL (Stage-B) ---
        model    mF1     mP     mR
    XGB a=0.0 0.5632 0.6809 0.5101
   XGB a=0.25 0.5863 0.6651 0.5450
    XGB a=0.5 0.5656 0.5873 0.5577
    XGB a=1.0 0.5595 0.5321 0.6097
XGB smote2000 0.5622 0.6404 0.5313
    RF a=0.25 0.5719 0.6459 0.5289
    ET a=0.25 0.5445 0.6022 0.5073

--- pairwise disagreement rate (v1 pool averaged ~0.12) ---
               XGB a=0.0  XGB a=0.25  XGB a=0.5  XGB a=1.0  XGB smote2000  RF a=0.25  ET a=0.25
XGB a=0.0          0.000       0.059      0.089      0.198          0.055      0.129      0.159
XGB a=0.25         0.059       0.000      0.074      0.173          0.062      0.132      0.163
XGB a=0.5          0.089       0.074      0.000      0.146          0.089      0.148      0.172
XGB a=1.0          0.198       0.173      0.146      0.000          0.191      0.215      0.229
XGB smote2000      0.055       0.062      0.089      0.191          0.000      0.129      0.158
RF a=0.25          0.129       0.132  

## 4. Combiners

- **Static equal** — the control that matters.
- **Global weights** $w \in \mathbb{R}^{M}$ — one scalar per model, coordinate ascent on macro-F1.
- **Per-class weights** $W \in \mathbb{R}^{M \times C}$ — the main proposal. Lets a recall-heavy
  member carry Shellcode while a precision-heavy member carries Exploits, which is exactly the split
  the α axis creates.
- **Stacking** — logistic regression and a shallow XGBoost over the concatenated base probabilities
  ($M \times C = 63$ meta-features), letting the meta-learner discover interactions the linear weight
  schemes cannot.

All are fitted on **DSEL**, which the base models never saw.

In [4]:
def fuse(probas, W):
    """W: (M,) or (M, C) -> fused probability matrix."""
    W = np.asarray(W, float)
    if W.ndim == 1:
        W = np.repeat(W[:, None], NC, axis=1)
    P = np.zeros_like(probas[0])
    for m in range(M):
        P += probas[m] * W[m][None, :]
    return P

def mf1(W, probas, ytrue):
    return f1_score(ytrue, fuse(probas, W).argmax(1), average="macro", zero_division=0)

GRID = (0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0)

def fit_global_weights(probas, ytrue, passes=4):
    W = np.ones(M) / M
    best = mf1(W, probas, ytrue)
    for _ in range(passes):
        for m in range(M):
            cur = W[m]
            for g in GRID:
                W[m] = g
                if W.sum() == 0:
                    continue
                s = mf1(W, probas, ytrue)
                if s > best:
                    best, cur = s, g
            W[m] = cur
    return W, best

def fit_per_class_weights(probas, ytrue, passes=3):
    W = np.ones((M, NC)) / M
    best = mf1(W, probas, ytrue)
    for _ in range(passes):
        for c in range(NC):
            for m in range(M):
                cur = W[m, c]
                for g in GRID:
                    W[m, c] = g
                    if W[:, c].sum() == 0:
                        continue
                    s = mf1(W, probas, ytrue)
                    if s > best:
                        best, cur = s, g
                W[m, c] = cur
    return W, best

t = time.time()
Wg, sg = fit_global_weights(P_dsel, yd)
Wc, sc = fit_per_class_weights(P_dsel, yd)
print("global weights   DSEL mF1 %.4f  (%.0fs)" % (sg, time.time() - t))
print("per-class weights DSEL mF1 %.4f" % sc)
print("\nglobal weights:", dict(zip(MODELS, Wg.round(2))))

global weights   DSEL mF1 0.5940  (3s)
per-class weights DSEL mF1 0.6278

global weights: {'XGB a=0.0': 0.5, 'XGB a=0.25': 1.5, 'XGB a=0.5': 0.25, 'XGB a=1.0': 0.25, 'XGB smote2000': 0.14, 'RF a=0.25': 0.5, 'ET a=0.25': 0.14}


In [5]:
# --- stacking meta-learners on the concatenated base probabilities ---
def meta_X(probas):
    return np.hstack(probas)

mw = sample_w(yd, WC25)
sc_meta = StandardScaler().fit(meta_X(P_dsel))

meta_lr = LogisticRegression(max_iter=2000, C=1.0, multi_class="multinomial")
meta_lr.fit(sc_meta.transform(meta_X(P_dsel)), yd, sample_weight=mw)

meta_xgb = make_xgb(n_estimators=200, max_depth=3, learning_rate=0.1)
meta_xgb.fit(meta_X(P_dsel), yd, sample_weight=mw)

print("meta-LR   DSEL mF1 %.4f (in-sample)"
      % f1_score(yd, meta_lr.predict(sc_meta.transform(meta_X(P_dsel))), average="macro", zero_division=0))
print("meta-XGB  DSEL mF1 %.4f (in-sample)"
      % f1_score(yd, meta_xgb.predict(meta_X(P_dsel)), average="macro", zero_division=0))
print("\n(in-sample on DSEL -- optimistic by construction; SEL below is the honest read)")

meta-LR   DSEL mF1 0.6275 (in-sample)
meta-XGB  DSEL mF1 0.9178 (in-sample)

(in-sample on DSEL -- optimistic by construction; SEL below is the honest read)


## 5. Variant comparison on SEL

End-to-end 10-class macro-F1 through the fixed gate is the headline. Single members and the static
average are the controls.

In [6]:
def score(fam_all, ye, gate_pred, tag):
    att    = (ye != 0).values
    ye_fam = ye[ye != 0].map(fam_to_idx).values
    yp = np.zeros(len(ye), int)
    g  = gate_pred == 1
    yp[g] = np.vectorize(idx_to_fam.get)(fam_all[g])
    return dict(variant=tag,
                stageB_mF1=round(f1_score(ye_fam, fam_all[att], average="macro", zero_division=0), 4),
                e2e_mF1=round(f1_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_mP=round(precision_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_mR=round(recall_score(ye, yp, average="macro", zero_division=0), 4),
                e2e_acc=round(float((yp == ye.values).mean()), 4))

VARIANTS = {
    "static: equal average":  lambda P: fuse(P, np.ones(M) / M).argmax(1),
    "global weights":         lambda P: fuse(P, Wg).argmax(1),
    "per-class weights":      lambda P: fuse(P, Wc).argmax(1),
    "stack: logistic":        lambda P: meta_lr.predict(sc_meta.transform(meta_X(P))),
    "stack: xgb(d=3)":        lambda P: meta_xgb.predict(meta_X(P)),
}
for i, nm in enumerate(MODELS):
    VARIANTS["single: " + nm] = (lambda ii: lambda P: P[ii].argmax(1))(i)

sel_res = [score(fn(P_sel), y_sel, gate_sel, nm) for nm, fn in VARIANTS.items()]
sel_df  = pd.DataFrame(sel_res).sort_values("e2e_mF1", ascending=False).reset_index(drop=True)
print(sel_df.to_string(index=False))

BESTV = sel_df.iloc[0]["variant"]
best_comb   = sel_df[~sel_df.variant.str.startswith("single")].iloc[0]
best_single_row = sel_df[sel_df.variant.str.startswith("single")].iloc[0]
static_row  = sel_df[sel_df.variant == "static: equal average"].iloc[0]
print("\nbest overall      : %-22s %.4f" % (BESTV, sel_df.iloc[0]["e2e_mF1"]))
print("best combiner     : %-22s %.4f" % (best_comb["variant"], best_comb["e2e_mF1"]))
print("best single member: %-22s %.4f" % (best_single_row["variant"], best_single_row["e2e_mF1"]))
print("combiner vs best single : %+.4f" % (best_comb["e2e_mF1"] - best_single_row["e2e_mF1"]))
print("combiner vs static avg  : %+.4f" % (best_comb["e2e_mF1"] - static_row["e2e_mF1"]))

              variant  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
    per-class weights      0.6277   0.6443  0.6762  0.6534   0.9043
       global weights      0.6154   0.6317  0.6809  0.6184   0.9083
      stack: logistic      0.6163   0.6310  0.6509  0.6304   0.9064
    single: XGB a=0.5      0.6180   0.6310  0.6385  0.6444   0.9073
static: equal average      0.6172   0.6301  0.6638  0.6244   0.9072
   single: XGB a=0.25      0.5995   0.6167  0.6589  0.6062   0.9078
    single: XGB a=0.0      0.5940   0.6130  0.6907  0.5843   0.9092
single: XGB smote2000      0.5939   0.6128  0.6760  0.6013   0.9087
      stack: xgb(d=3)      0.5853   0.6106  0.6706  0.5848   0.9079
    single: RF a=0.25      0.5782   0.5984  0.6616  0.5709   0.9025
    single: XGB a=1.0      0.5758   0.5917  0.5534  0.6654   0.8958
    single: ET a=0.25      0.5665   0.5889  0.6314  0.5672   0.9006

best overall      : per-class weights      0.6443
best combiner     : per-class weights      0.6443
best single mem

### 5.1 Which member wins which class?

The per-class weight matrix, normalised per column. If the α axis worked, the recall-heavy members
(α=1, SMOTE) should dominate the rare families and the precision-heavy ones (α=0, α=0.25) the large
families. If instead one column pattern repeats everywhere, the pool is still not diverse.

In [7]:
Wn = Wc / np.maximum(Wc.sum(0, keepdims=True), 1e-9)
print(pd.DataFrame(Wn, index=MODELS, columns=FAM).round(3).to_string())
print("\ndominant member per family:")
for c, f in enumerate(FAM):
    print("  %-15s %s (%.2f)" % (f, MODELS[Wn[:, c].argmax()], Wn[:, c].max()))

               Analysis  Backdoor    DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
XGB a=0.0         0.467     0.467  0.259     0.143    0.064    0.329           0.609      0.523  0.105
XGB a=0.25        0.311     0.133  0.074     0.143    0.257    0.082           0.152      0.262  0.105
XGB a=0.5         0.044     0.000  0.074     0.143    0.064    0.247           0.076      0.065  0.105
XGB a=1.0         0.044     0.000  0.389     0.143    0.514    0.082           0.076      0.037  0.368
XGB smote2000     0.044     0.133  0.130     0.143    0.064    0.047           0.043      0.037  0.105
RF a=0.25         0.044     0.133  0.000     0.143    0.037    0.047           0.000      0.037  0.105
ET a=0.25         0.044     0.133  0.074     0.143    0.000    0.165           0.043      0.037  0.105

dominant member per family:
  Analysis        XGB a=0.0 (0.47)
  Backdoor        XGB a=0.0 (0.47)
  DoS             XGB a=1.0 (0.39)
  Exploits        XGB a=0.0 (0.14)
  Fuzze

## 6. Final evaluation — test set, once

The SEL-selected variant, applied to test. Reference points: the train-only single α=0.25 XGBoost
(like-for-like, 0.6125) and the weighting study's train+val refit (0.6157, for context).

In [8]:
fam_test = VARIANTS[BESTV](P_test)
final    = score(fam_test, y_test, gate_test, BESTV)
ref      = score(P_test[MODELS.index("XGB a=0.25")].argmax(1), y_test, gate_test,
                 "single: XGB a=0.25 (train-only)")
print("--- TEST ---")
print(pd.DataFrame([final, ref]).to_string(index=False))
print("\nensemble vs train-only single XGB a=0.25 : %+.4f" % (final["e2e_mF1"] - ref["e2e_mF1"]))
print("vs weighting-study best (0.6157, train+val refit): %+.4f" % (final["e2e_mF1"] - 0.6157))
print("vs DES v1 best on test (0.6118): %+.4f" % (final["e2e_mF1"] - 0.6118))

--- TEST ---
                        variant  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
              per-class weights      0.6146   0.6302  0.6733  0.6330   0.9030
single: XGB a=0.25 (train-only)      0.5966   0.6125  0.6580  0.6002   0.9068

ensemble vs train-only single XGB a=0.25 : +0.0177
vs weighting-study best (0.6157, train+val refit): +0.0145
vs DES v1 best on test (0.6118): +0.0184


In [9]:
yp = np.zeros(len(y_test), int)
g  = gate_test == 1
yp[g] = np.vectorize(idx_to_fam.get)(fam_test[g])
print(classification_report(y_test, yp, labels=list(range(10)),
                            target_names=[LABELS[i] for i in range(10)], digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_test, yp, labels=list(range(10))),
                   index=[LABELS[i] for i in range(10)],
                   columns=[LABELS[i] for i in range(10)]).to_string())

                precision    recall  f1-score   support

        Benign      0.999     0.968     0.983     45001
      Analysis      0.318     0.649     0.427        77
      Backdoor      0.938     0.500     0.652        90
           DoS      0.525     0.368     0.433       883
      Exploits      0.900     0.642     0.750      6182
       Fuzzers      0.574     0.926     0.708      5279
       Generic      0.768     0.814     0.790       899
Reconnaissance      0.623     0.661     0.641      2406
     Shellcode      0.379     0.343     0.360       370
         Worms      0.710     0.458     0.557        48

      accuracy                          0.903     61235
     macro avg      0.673     0.633     0.630     61235
  weighted avg      0.922     0.903     0.906     61235

                Benign  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Benign           43547        39         2   18        60     1171       14             131         19 

### 6.1 All variants on test — ablation appendix

Selection already happened on SEL. This table exists to show whether the SEL ranking **transferred**
(the thing v1 failed) and **must not** be used to pick a different winner after the fact.

In [10]:
test_all = (pd.DataFrame([score(fn(P_test), y_test, gate_test, nm) for nm, fn in VARIANTS.items()])
            .sort_values("e2e_mF1", ascending=False).reset_index(drop=True))
print(test_all.to_string(index=False))

rank = 1 + list(test_all.variant).index(BESTV)
print("\nSEL-selected variant '%s' ranks %d of %d on test" % (BESTV, rank, len(test_all)))
sel_order  = list(sel_df.variant)
test_order = list(test_all.variant)
rho = pd.Series([sel_order.index(v) for v in test_order]).corr(pd.Series(range(len(test_order))), method="spearman")
print("Spearman rank correlation SEL vs test: %.3f  (high = selection protocol is trustworthy)" % rho)

              variant  stageB_mF1  e2e_mF1  e2e_mP  e2e_mR  e2e_acc
    per-class weights      0.6146   0.6302  0.6733  0.6330   0.9030
       global weights      0.6026   0.6184  0.6680  0.6044   0.9068
static: equal average      0.6008   0.6149  0.6553  0.6062   0.9059
   single: XGB a=0.25      0.5966   0.6125  0.6580  0.6002   0.9068
    single: XGB a=0.5      0.5958   0.6092  0.6114  0.6243   0.9044
      stack: logistic      0.5904   0.6074  0.6308  0.6040   0.9037
    single: XGB a=0.0      0.5846   0.6025  0.6825  0.5762   0.9071
single: XGB smote2000      0.5760   0.5952  0.6550  0.5824   0.9064
      stack: xgb(d=3)      0.5688   0.5930  0.6567  0.5651   0.9049
    single: RF a=0.25      0.5580   0.5780  0.6332  0.5555   0.9011
    single: XGB a=1.0      0.5587   0.5740  0.5325  0.6556   0.8940
    single: ET a=0.25      0.5384   0.5597  0.5979  0.5420   0.8990

SEL-selected variant 'per-class weights' ranks 1 of 12 on test
Spearman rank correlation SEL vs test: 0.937  (high 

## 7. Findings

**Positive result: per-class weighted fusion over an α-diverse pool reaches test macro-F1 0.6302 —
the best number in the project, and it transferred cleanly from the selection split.**

| Configuration | SEL | **Test** |
|---|---|---|
| **Per-class weights (α-diverse pool)** | **0.6443** | **0.6302** |
| Global weights (7 params) | 0.6317 | 0.6184 |
| Static equal average | 0.6301 | 0.6149 |
| Single XGB α=0.25 (train-only) | 0.6167 | 0.6125 |
| Stack: logistic | 0.6310 | 0.6074 |
| Stack: XGB(d=3) | 0.6106 | 0.5930 |

- **+0.0177** over the like-for-like train-only single model
- **+0.0145** over the weighting study's train+val refit (0.6157)
- **+0.0184** over the DES v1 best (0.6118)

The SEL-selected variant ranks **1st of 12 on test**, and the SEL↔test Spearman rank correlation is
**0.937**. Contrast v1, where the SEL winner fell to rank 2 and the gain vanished: this time the
selection protocol held, which is what makes the number quotable.

### 1. The α axis was the missing diversity — v1's conclusion was wrong

The DES notebook concluded that ensembling was closed for Stage B because RF/ET/XGB were not
complementary. That conclusion was **too broad**: it was true of a pool diversified by *algorithm*,
and false of one diversified by *class weighting*.

| Pool | mean pairwise disagreement | oracle macro-F1 | headroom over best single |
|---|---|---|---|
| v1 (XGB / RF / ET, all α=0.25) | ~0.12 | 0.6799 | 0.0804 |
| **v2 (α-diverse)** | **0.135** | **0.7318** | **0.1456** |

Oracle headroom nearly **doubled** and at-least-one-model-correct rose from 79.1% to 83.5%. The
disagreement matrix localises the source precisely: **XGB α=1.0 disagrees with the rest on 15–23% of
rows**, while the α=0/0.25/SMOTE members cluster tightly at 5–9%. One deliberately recall-heavy member
supplied most of the usable diversity, and it is a member that is *individually poor* (macro-F1
0.5595, worst but one). That is the ensemble lesson: pool members should be selected for
complementarity, not individual quality.

### 2. The per-class weight matrix specialises exactly as designed

| Family | Dominant member | Weight |
|---|---|---|
| DoS | XGB α=1.0 | 0.39 |
| Fuzzers | XGB α=1.0 | 0.51 |
| Worms | XGB α=1.0 | 0.37 |
| Analysis | XGB α=0.0 | 0.47 |
| Backdoor | XGB α=0.0 | 0.47 |
| Reconnaissance | XGB α=0.0 | 0.61 |
| Shellcode | XGB α=0.0 | 0.52 |
| Generic | XGB α=0.0 | 0.33 |
| Exploits | *(unchanged, 1/7 uniform)* | 0.14 |

The matrix splits the pool along the precision↔recall axis, family by family — the recall-heavy α=1
member takes DoS/Fuzzers/Worms, the precision-heavy α=0 member takes the families that recall-heavy
models over-predict. This is genuine per-class specialisation, not the near-uniform weights that made
v1 a static ensemble in disguise (0.333 ± 0.02 there).

Two details worth reading carefully:

- **Exploits kept its initialisation weights** (all members at 1/7). Coordinate ascent could not
  improve on a plain average for the largest, easiest family — sensible, and a good sign the search
  is not simply overfitting every free parameter it is given.
- **The rare families went to the *precision*-heavy member, not the recall-heavy one** — the opposite
  of the naive expectation. It is the right behaviour: Analysis/Shellcode/Backdoor are already
  *over*-predicted (their precision, not their recall, is the binding constraint), so macro-F1 is
  maximised by damping them. The combiner found this from data.

### 3. Stacking failed where linear per-class weighting succeeded

Both meta-learners came in **below the single α=0.25 model** on test (LR 0.6074, XGB 0.5930). The
meta-XGB scored **0.9178 in-sample on DSEL** against 0.5930 on test — a textbook overfit: a depth-3
booster on 6,494 rows with 63 meta-features memorises the combination instead of learning it.

The instructive contrast is with the per-class matrix, which has 63 parameters too and *did*
generalise. The difference is inductive bias, not capacity: the matrix is constrained to a
non-negative linear reweighting of calibrated probabilities, while the meta-learners are free to
invent arbitrary decision surfaces over them. **Constrained combiners beat flexible ones when the
combiner-fitting set is small** — and DSEL contributes only ~20–30 rows for the rarest families.

### 4. What the gain is made of

Per-class test F1 against the α=0.25 single model:

| Family | Single α=0.25 | **Ensemble** | Δ |
|---|---|---|---|
| Worms | 0.415 | **0.557** | **+0.142** |
| Backdoor | 0.599 | **0.652** | +0.053 |
| DoS | 0.411 | **0.433** | +0.022 |
| Reconnaissance | 0.641 | 0.641 | 0.000 |
| Shellcode | 0.363 | 0.360 | −0.003 |
| Fuzzers | 0.713 | 0.708 | −0.005 |
| Analysis | 0.439 | 0.427 | −0.012 |
| Generic | 0.813 | 0.790 | −0.023 |
| Exploits | 0.782 | 0.750 | −0.032 |

The ensemble **trades large-family F1 for rare-family F1** — which is precisely what a macro-F1
objective should do, and precisely the literature gap this thesis targets. Exploits pays the largest
share (precision rises to 0.900 but recall falls to 0.642, with 1,195 Exploits going to Fuzzers and
554 to Reconnaissance). Whether that trade is acceptable is a SOC decision, and it should be stated
as one rather than buried in a macro average.

Only ~12% of the 0.1456 oracle headroom was realised. Substantial complementarity remains
unexploited — but unlike v1, it is now demonstrably reachable in part.

### 5. Data-usage note (keeps the comparison fair)

The ensemble trains base models on `train` (195,950) and fits the combiner on DSEL (half of val).
The 0.6157 reference was a single model refit on **train+val** (244,938). The ensemble therefore uses
*less* data overall, so the +0.0145 is not an artifact of extra training rows. Refitting the pool on
train+DSEL with SEL as the combiner-fitting set is an obvious follow-up and would likely add a little
more.

### Recommendation

**Adopt this as the Stage-B model.** New project best: **0.6302**.

Cost is honest and modest: seven XGBoost-class models at inference plus a 7×9 weight matrix — no kNN
query per flow, no DSEL resident in memory (unlike v1), and the members are independent so they
parallelise.

### Next levers

1. **Per-class decision thresholds** on the fused probabilities — still untried, and now applied to a
   better-calibrated starting point. Fuzzers (P 0.574 / R 0.926) and Exploits (P 0.900 / R 0.642) are
   badly off-balance in *opposite* directions, which is close to ideal for threshold tuning.
2. **Refit the pool on train+DSEL**, combiner on SEL (see §5).
3. **Widen the α grid** — α=1.0 supplied most of the diversity, so α=1.5/2.0 (beyond full inverse
   frequency) may add more even though they are poor standalone models.
4. Gate threshold, soft routing, label-ambiguity ceiling, SHAP.
